# Condition Shift Baseline Notebook

이 노트북은 `condition_shift_baseline` 실험의 thin orchestrator다.

- Colab/서버에서 Git checkout 상태를 확인한다.
- 필요하면 prepared dataset bootstrap script를 호출한다.
- versioned Python runner를 실행한다.
- `summary.json`과 `log.txt`를 읽어 표와 간단 시각화를 보여준다.

실험 핵심 로직은 노트북에 두지 않고, notebook이 호출하는 `.py`는 repo에 versioned 상태로 유지한다.


## 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys
import time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents, Path('/content/ReGraM')]
REPO_ROOT = next((p.resolve() for p in repo_candidates if p.exists() and p.name in {'ReGraM'}), cwd)
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
REPORT_ROOT = EXP_ROOT / 'reports'
NOTEBOOK_ROOT = EXP_ROOT / 'notebook'
MANIFEST_ROOT_CANDIDATES = [REPO_ROOT / 'manifests', EXP_ROOT / 'manifests']

def now_string():
    return datetime.now().astimezone().strftime('%Y-%m-%d %H:%M:%S %Z')

def format_run_label(config):
    return config['category']

def display_title(title, body=None):
    text = f"## {title}"
    if body:
        text += f"{body}"
    display(Markdown(text))

def display_run_plan(run_configs):
    rows = []
    for idx, config in enumerate(run_configs, start=1):
        rows.append({
            'order': idx,
            'category': config['category'],
            'manifest_count': len(config['manifest_paths']),
            'summary_name': config['summary_path'].name,
            'wandb': 'on' if config.get('use_wandb') else 'off',
        })
    display(pd.DataFrame(rows))

def display_environment_summary():
    display_title('Environment Summary')
    display(pd.DataFrame([
        {'key': 'REPO_ROOT', 'value': str(REPO_ROOT)},
        {'key': 'EXP_ROOT', 'value': str(EXP_ROOT)},
        {'key': 'REPORT_ROOT', 'value': str(REPORT_ROOT)},
    ]))

run_history = []

print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Git Checkout

Colab에서는 먼저 repo를 Git으로 clone 또는 pull 해서 notebook이 사용할 `.py`를 최신 상태로 맞춘다.


In [ ]:
REPO_URL = 'https://github.com/outSeop/ReGraM.git'
REPO_DIR = Path('/content/ReGraM')
git_cmd = (
    f'if [ -d "{REPO_DIR}/.git" ]; then git -C "{REPO_DIR}" pull --ff-only; '
    f'else git clone "{REPO_URL}" "{REPO_DIR}"; fi'
)
print(git_cmd)
subprocess.run(['bash', '-lc', git_cmd], check=True)

REPO_ROOT = REPO_DIR.resolve()
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
REPORT_ROOT = EXP_ROOT / 'reports'
NOTEBOOK_ROOT = EXP_ROOT / 'notebook'
MANIFEST_ROOT_CANDIDATES = [REPO_ROOT / 'manifests', EXP_ROOT / 'manifests']

print('updated REPO_ROOT =', REPO_ROOT)
print('updated EXP_ROOT =', EXP_ROOT)


## Dataset load

In [ ]:
from pathlib import Path
import subprocess

drive_tar = Path("/content/drive/MyDrive/ReGraM/data/row/mvtec_loco_anomaly_detection.tar.gz")
runtime_tar = Path("/content/mvtec_loco_anomaly_detection.tar.gz")
runtime_row = Path("/content/ReGraM/data/row")
runtime_root = runtime_row / "mvtec_loco_anomaly_detection"

print("drive_tar exists:", drive_tar.exists())
print("runtime_root exists:", runtime_root.exists())

runtime_row.mkdir(parents=True, exist_ok=True)

if not runtime_root.exists():
    if not runtime_tar.exists():
        subprocess.run(["cp", str(drive_tar), str(runtime_tar)], check=True)
    subprocess.run(
        ["tar", "-xf", str(runtime_tar), "-C", str(runtime_row)],
        check=True,
    )

print("done")
print(runtime_root.exists(), runtime_root)


## Optional Dataset Bootstrap

Git checkout 이후 prepared dataset이나 작은 reports 자산이 필요하면 아래처럼 별도 Python bootstrap script를 호출한다.
코드 동기화는 bootstrap이 아니라 Git이 담당한다.


In [ ]:
bootstrap_cmd = [
    sys.executable,
    str(EXP_ROOT / 'colab' / 'bootstrap_runtime.py'),
    '--dry-run',
]
print(' '.join(bootstrap_cmd))
subprocess.run(bootstrap_cmd, cwd=REPO_ROOT, check=True)


## Runner Orchestration

노트북은 어떤 runner를 어떤 인자로 호출할지 만 관리한다.


## Runner Config

manifest와 category를 먼저 정하고, runner와 summary viewer가 같은 설정을 공유하게 둔다.


In [ ]:
RUNNER_NAME = 'PatchCore manifest shift evaluation'
RUNNER_PATH = EXP_ROOT / 'src' / 'core' / 'run_patchcore_manifest_shift.py'
RUNNER_INPUT_DESC = 'category + manifest jsonl(s) + raw LOCO dataset root'
RUNNER_OUTPUT_DESC = 'single summary json per category under reports/patchcore_manifest_shift'

CATEGORIES = ['breakfast_box']
AUTO_DISCOVER_MANIFESTS = True
MANIFEST_NAMES = [
    'query_motion_blur.jsonl',
    'query_low_light.jsonl',
    'query_gaussian_noise.jsonl',
]
EXCLUDED_MANIFEST_NAMES = {'query_identity.jsonl', 'query_multi.jsonl'}
USE_WANDB = True
WANDB_PROJECT = 'regram-condition-shift'
WANDB_GROUP = 'patchcore'
WANDB_MODE = 'online'  # online | offline | disabled
WANDB_LOG_IMAGES = True
WANDB_MAX_IMAGES = 2
DEVICE = 'cuda'

if AUTO_DISCOVER_MANIFESTS:
    discovered = []
    for root in MANIFEST_ROOT_CANDIDATES:
        if root.exists():
            discovered.extend(path.name for path in sorted(root.glob('query_*.jsonl')))
    MANIFEST_NAMES = sorted({name for name in discovered if name not in EXCLUDED_MANIFEST_NAMES})

manifest_paths = []
for manifest_name in MANIFEST_NAMES:
    manifest_path = next(
        (root / manifest_name for root in MANIFEST_ROOT_CANDIDATES if (root / manifest_name).exists()),
        None,
    )
    if manifest_path is None:
        searched = [str(root / manifest_name) for root in MANIFEST_ROOT_CANDIDATES]
        raise FileNotFoundError(f'Manifest not found. searched={searched}')
    manifest_paths.append(manifest_path)

run_configs = []
for category in CATEGORIES:
    summary_path = REPORT_ROOT / 'patchcore_manifest_shift' / f'{category}_multi_all.json'
    runner_cmd = [
        sys.executable,
        str(RUNNER_PATH),
        '--category', category,
        '--manifest', *[str(p) for p in manifest_paths],
        '--device', DEVICE,
    ]
    if USE_WANDB:
        runner_cmd.extend([
            '--use-wandb',
            '--wandb-project', WANDB_PROJECT,
            '--wandb-group', WANDB_GROUP,
            '--wandb-mode', WANDB_MODE,
        ])
        if WANDB_LOG_IMAGES:
            runner_cmd.extend([
                '--wandb-log-images',
                '--wandb-max-images', str(WANDB_MAX_IMAGES),
            ])
    run_configs.append({
        'category': category,
        'manifest_paths': list(manifest_paths),
        'manifest_names': list(MANIFEST_NAMES),
        'summary_path': summary_path,
        'runner_cmd': runner_cmd,
        'use_wandb': USE_WANDB,
    })

display_title(
    'Runner Plan',
    f'Prepared `{len(run_configs)}` PatchCore runs across `{len(CATEGORIES)}` categories '
    f'(each run consumes `{len(MANIFEST_NAMES)}` manifests).',
)
display(pd.DataFrame([
    {'key': 'runner_name', 'value': RUNNER_NAME},
    {'key': 'runner_path', 'value': str(RUNNER_PATH)},
    {'key': 'runner_inputs', 'value': RUNNER_INPUT_DESC},
    {'key': 'runner_outputs', 'value': RUNNER_OUTPUT_DESC},
    {'key': 'categories', 'value': ','.join(CATEGORIES)},
    {'key': 'auto_discover_manifests', 'value': AUTO_DISCOVER_MANIFESTS},
    {'key': 'manifest_count', 'value': len(MANIFEST_NAMES)},
    {'key': 'device', 'value': DEVICE},
    {'key': 'wandb_project', 'value': WANDB_PROJECT if USE_WANDB else 'off'},
    {'key': 'wandb_group', 'value': WANDB_GROUP if USE_WANDB else 'off'},
    {'key': 'wandb_mode', 'value': WANDB_MODE if USE_WANDB else 'disabled'},
    {'key': 'wandb_log_images', 'value': WANDB_LOG_IMAGES if USE_WANDB else False},
    {'key': 'wandb_max_images', 'value': WANDB_MAX_IMAGES if USE_WANDB else 0},
]))
display(pd.DataFrame({'manifest_name': MANIFEST_NAMES}))
display_run_plan(run_configs)


In [ ]:
display_title('Run Command Preview', 'Each run covers one category with all discovered manifests in a single process.')
for config in run_configs:
    print('---')
    print('label =', format_run_label(config))
    print('manifest_count =', len(config['manifest_paths']))
    print('summary_path =', config['summary_path'])
    print('runner_cmd =', ' '.join(config['runner_cmd']))


In [ ]:
!mkdir -p /content/ReGraM/external
!test -d /content/ReGraM/external/patchcore-inspection.clean/.git || git clone https://github.com/amazon-science/patchcore-inspection.git /content/ReGraM/external/patchcore-inspection.clean
!pip install faiss-cpu timm


In [ ]:
run_history = []
total_runs = len(run_configs)
display_title('Execution Log', f'Started at `{now_string()}` with `{total_runs}` scheduled runs.')

for idx, config in enumerate(run_configs, start=1):
    label = format_run_label(config)
    started_at = now_string()
    print(f"[{idx}/{total_runs}] START {label} @ {started_at}")
    print('summary_path =', config['summary_path'])
    print('command =', ' '.join(config['runner_cmd']))
    tic = time.perf_counter()
    result = subprocess.run(
        config['runner_cmd'],
        cwd=REPO_ROOT,
        text=True,
    )
    elapsed_sec = round(time.perf_counter() - tic, 2)
    status = 'success' if result.returncode == 0 else 'failed'
    finished_at = now_string()
    run_history.append({
        'order': idx,
        'category': config['category'],
        'status': status,
        'returncode': result.returncode,
        'elapsed_sec': elapsed_sec,
        'started_at': started_at,
        'finished_at': finished_at,
        'summary_path': str(config['summary_path']),
    })
    print(f"[{idx}/{total_runs}] END   {label} -> {status} ({elapsed_sec}s) @ {finished_at}")
    if result.returncode != 0:
        display(pd.DataFrame(run_history))
        raise RuntimeError(f"runner failed for {label}")

display_title('Execution Summary', f'Finished at `{now_string()}`.')
display(pd.DataFrame(run_history))


## Summary Viewer

runner가 남긴 `summary.json`과 `log.txt`를 기준으로만 결과를 본다.


In [ ]:
rows = []
clean_rows = []
for config in run_configs:
    summary_path = config['summary_path']
    if not summary_path.exists():
        display(Markdown(f"`summary.json` not found: `{summary_path}`"))
        continue

    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    payload = summary.get('payload', {})
    clean_rows.append({
        'category': summary['class_name'],
        'clean_image_auroc': summary['metrics'].get('clean_image_auroc'),
        'clean_good_mean': summary['metrics'].get('clean_good_mean'),
        'clean_anomaly_mean': summary['metrics'].get('clean_anomaly_mean'),
    })
    severity_spec_by_cell = payload.get('severity_spec_by_cell', {})
    for aug_type, severity_map in payload.get('augmentations', {}).items():
        for severity, item in severity_map.items():
            cell_key = f'{aug_type}/{severity}'
            spec = severity_spec_by_cell.get(cell_key, payload.get('severity_spec', {}))
            rows.append({
                'category': summary['class_name'],
                'shift_family': aug_type,
                'severity': severity,
                'mean': item.get('mean'),
                'fpr_over_clean_max': item.get('fpr_over_clean_max'),
                'mean_score_shift': item.get('mean_score_shift'),
                'image_auroc_vs_clean_anomaly': item.get('image_auroc_vs_clean_anomaly'),
                **{f'severity_param_{k}': v for k, v in spec.items()},
            })

df = pd.DataFrame(rows).sort_values(['category', 'shift_family', 'severity']).reset_index(drop=True)
clean_df = pd.DataFrame(clean_rows).drop_duplicates(subset=['category']).sort_values(['category']).reset_index(drop=True)
display_title('Summary Viewer', 'Per-category summary with every (shift, severity) cell expanded.')
display(df)

display_title('Clean Reference Snapshot', 'Per-category clean baseline metrics copied from each summary.')
display(clean_df)

if run_history:
    display_title('Run History Snapshot')
    display(pd.DataFrame(run_history))


In [ ]:
severity_order = {'low': 0, 'medium': 1, 'high': 2}
plot_df = df.copy()
plot_df['severity_rank'] = plot_df['severity'].map(severity_order)
plot_df = plot_df.sort_values(['shift_family', 'severity_rank']).reset_index(drop=True)

metric_specs = [
    ('fpr_over_clean_max', 'Shifted FPR over clean max', 'FPR'),
    ('mean_score_shift', 'Shifted mean score shift', 'Score shift'),
    ('image_auroc_vs_clean_anomaly', 'Shifted vs anomaly AUROC', 'AUROC'),
]

display_title('Metric Tables', 'Pivoted summary tables across all 8 shifts × 3 severities (mean across categories if >1).')
for metric_key, metric_title, _ in metric_specs:
    pivot = plot_df.pivot_table(index='shift_family', columns='severity', values=metric_key, aggfunc='mean')
    ordered_cols = [col for col in ['low', 'medium', 'high'] if col in pivot.columns]
    pivot = pivot[ordered_cols]
    display(Markdown(f'### {metric_title}'))
    display(pivot)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
for ax, (metric_key, metric_title, y_label) in zip(axes, metric_specs):
    pivot = plot_df.pivot_table(index='shift_family', columns='severity', values=metric_key, aggfunc='mean')
    ordered_cols = [col for col in ['low', 'medium', 'high'] if col in pivot.columns]
    pivot = pivot[ordered_cols]
    pivot.plot(kind='bar', ax=ax)
    ax.set_title(metric_title)
    ax.set_xlabel('shift_family')
    ax.set_ylabel(y_label)
    ax.legend(title='severity')
    ax.tick_params(axis='x', rotation=45)
plt.show()


In [ ]:
display_title('Recommended W&B Panels', 'Suggested panel recipes for the per-category multi-shift run schema.')
panel_specs = pd.DataFrame([
    {
        'panel_name': 'Worst FPR across all shift cells',
        'metric': 'worst_fpr_over_clean_max',
        'filter': 'project=regram-condition-shift',
        'group_by': 'group (= baseline), config.class_name',
        'notes': 'Sort Runs page by this scalar to see which (baseline, category) combo breaks hardest.',
    },
    {
        'panel_name': 'Mean FPR by severity',
        'metric': 'mean_fpr_by_severity/low, mean_fpr_by_severity/medium, mean_fpr_by_severity/high',
        'filter': 'project=regram-condition-shift',
        'group_by': 'group',
        'notes': 'Line chart of low/medium/high progression averaged over all 8 shift families.',
    },
    {
        'panel_name': 'Shift cells detail table',
        'metric': 'shift_cells (wandb.Table)',
        'filter': 'open individual run',
        'group_by': 'native table: shift, severity',
        'notes': 'Sort/filter 24 cells inside the run. Use wandb Compare mode to join tables across baselines.',
    },
    {
        'panel_name': 'Clean image AUROC across baselines',
        'metric': 'clean_image_auroc',
        'filter': 'project=regram-condition-shift',
        'group_by': 'group, config.class_name',
        'notes': 'Reference scalar — should be stable across runs of the same (baseline, category) pair.',
    },
    {
        'panel_name': 'Worst cell identity',
        'metric': 'worst_cell (string)',
        'filter': 'project=regram-condition-shift',
        'group_by': 'column in Runs table',
        'notes': 'Shows which shift/severity caused the worst FPR — quickest diagnostic column.',
    },
])
display(panel_specs)
